# External review: seed repeats and generalisation runs

Answers two comments of the external review (`docs/external_review_response.md`):

- **Comment 1** - the phone-line results repeated over random seeds, so stability is
  measured rather than assumed.
- **Comment 2** - three opt-in training changes aimed at unseen-tool generalisation:
  a fresh random crop each epoch, augmentation that randomises the recording cues the
  shortcut floor found, and a learned mix of all encoder layers.

**Selection rule, fixed before any result is seen.** Among the published recipe and
the four variants, the one with the lowest **dev** EER wins; dev EERs within 0.5
points count as a tie, which goes to the variant with fewer changes. The test sets
(eval pool, CM04 Tortoise, RVC) are scored for every run and reported in full, but
never used to choose.

**Blocks, in order** (each run is saved the moment it finishes):

| Block | Runs | Time on a T4 |
|---|---|---|
| deployed_seeds | deployed S2, seeds 1234 (re-run), 2025, 7, 42, 99 | ~1 h |
| variants | deployed recipe + crop / +augment / +layer mix / all three | ~1 h |
| s3_seeds | S3 XTTS-only phone line, seeds 2025, 7, 42, 99 | ~1.7 h |
| xtts_channel_seeds | S2 XTTS-only phone line, seeds 2025, 7, 42, 99 | ~0.8 h |
| best_variant_seeds | the dev-selected variant, seeds 2025, 7 | ~0.5 h |

**Getting the results.** When the run ends, the last cell (section 5) shows a link:
click it and `review_results.zip` downloads. Do this **before** closing the tab or
stopping the session -- an interactive session's files are deleted when it ends.
Optional backup: with the secret `KAGGLE_KEY` attached (Add-ons -> Secrets), the zip is
also copied after every block to the private dataset
`saikrishnareddy9/codemix-review-results`. Without it everything else still runs.

**How to run.** Inputs: `saikrishnareddy9/codemix-bundle-normalised` (required) and
`saikrishnareddy9/codemix-s3-native-results` (recommended: it holds the Stage-1
checkpoint). Settings: GPU T4, Internet On. Use **Run All in the editor** (committed
runs currently fail to mount the dataset). Keep the tab open. A failing step is
recorded and the run continues; `RUN HEALTH` heads the summary.

## 1. Settings

In [ ]:
SMOKE = False      # True -> ~20 min check of every block with tiny runs; numbers mean nothing

BLOCKS = ["deployed_seeds", "variants", "s3_seeds", "xtts_channel_seeds", "best_variant_seeds"]
SEEDS = [2025, 7, 42, 99]            # extra seeds; the published runs used 1234
VARIANTS = {                          # opt-in options, added to the deployed recipe
    "crop": {"crop_per_epoch": True},
    "crop_aug": {"crop_per_epoch": True, "augment": "channel_v1"},
    "crop_layers": {"crop_per_epoch": True, "layer_weighting": True},
    "crop_aug_layers": {"crop_per_epoch": True, "augment": "channel_v1", "layer_weighting": True},
}
TIE = 0.005                           # dev-EER ties within half a point go to the simpler variant
#: Candidates finished in an earlier run, which the selection rule must also see. Their
#: test-set EERs come along for the report only; choosing uses dev EER alone.
PRIOR = []
#: explore_seeds block: the variant (a key of VARIANTS) to repeat over SEEDS. It is
#: chosen after seeing test scores, so its results are exploratory, never a selection.
EXPLORE = None

# ---- nothing below should need changing ------------------------------------------
REPO_HOST = "github.com/Mounika-Reddy-0802/codemix-deepfake-detection.git"
BRANCH = "week12-krishna-paper-chapters"
#: After every block the results zip is copied to this PRIVATE dataset on your account,
#: so a watcher on the laptop can download it while nobody is looking at this tab.
#: Needs the secret KAGGLE_KEY (Add-ons -> Secrets); without it the run still works.
RESULTS_DATASET = "saikrishnareddy9/codemix-review-results"
BASE = {                              # system -> (published config, published result prefix)
    "deployed": ("configs/train_lora_norm_rvc_channel.yaml", "results/w10/lora_norm_rvc_channel"),
    "xtts_channel": ("configs/train_lora_norm_channel.yaml", "results/w10/lora_norm_channel"),
    "s3_xtts_channel": ("configs/train_s3_native_channel.yaml",
                        "docs/results/s3_native/s3_native_channel"),
}
EVALS = {
    "eval_pool": "data/manifests/codemix_eval_channel20.csv",
    "cm04": "data/manifests/score_cm04_norm_channel20.csv",
    "rvc_test": "data/manifests/rvc_holdout_test_channel20.csv",
}
SCORE_ARGS = "--device cuda --batch-size 32 --num-workers 4 --max-seconds 4.0"

## 2. Setup: clone, Stage-1 checkpoint, data check, config check

In [ ]:
import glob, hashlib, json, os, re, shutil, subprocess, sys, time, traceback
from pathlib import Path

WORK = Path("/kaggle/working")
REPO = WORK / "codemix-deepfake-detection"
OUT = WORK / "review_results"


def sh(cmd, check=True):
    """Run a short command and show its output."""
    print("$", cmd)
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True,
                       cwd=REPO if REPO.exists() else None)
    if r.stdout.strip():
        print(r.stdout[-2500:])
    if r.returncode and r.stderr.strip():
        print(r.stderr[-2500:])
    if check and r.returncode:
        raise SystemExit(f"failed ({r.returncode}): {cmd}")
    return r


def run_logged(args):
    """Run a long command, streaming its output live, and return all of it."""
    args = [str(a) for a in args]
    print("$", " ".join(args), flush=True)
    proc = subprocess.Popen(args, text=True, errors="replace", cwd=REPO,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    lines = []
    with open(OUT / "run_log.txt", "a", encoding="utf-8") as log:  # the full output is kept
        log.write(f"\n$ {' '.join(args)}\n")
        for line in proc.stdout:
            lines.append(line)
            log.write(line)
            if not any(noise in line for noise in ("Warning", "warn(", "Some weights", "You should")):
                print(line, end="", flush=True)
    proc.wait()
    return proc.returncode, "".join(lines)


def _make_writable(func, path, _):
    """git marks its pack files read-only; clear that so a re-run can remove them."""
    os.chmod(path, 0o700)
    func(path)


# Skip LFS on clone: a normal clone would download every checkpoint in the repo.
os.environ["GIT_LFS_SKIP_SMUDGE"] = "1"
for attempt in range(1, 4):  # GitHub occasionally drops a connection
    if REPO.exists():
        shutil.rmtree(REPO, onerror=_make_writable)
    if subprocess.run(["git", "clone", "--branch", BRANCH, "--depth", "1",
                       f"https://{REPO_HOST}", str(REPO)]).returncode == 0:
        break
    print(f"clone attempt {attempt} failed; retrying in 20 s", flush=True)
    time.sleep(20)
else:
    raise SystemExit("Could not clone the repository after 3 attempts. Is Internet switched On?")
os.chdir(REPO)
sys.path.insert(0, str(REPO))
shutil.rmtree(OUT, ignore_errors=True)
(WORK / "review_results.zip").unlink(missing_ok=True)
OUT.mkdir()
COMMIT = subprocess.run(["git", "rev-parse", "HEAD"], cwd=REPO, text=True,
                        capture_output=True).stdout.strip()


def save_progress():
    """Zip the results so far (no checkpoints) into /kaggle/working/review_results.zip."""
    import zipfile
    staging = WORK / "review_results.zip.tmp"
    with zipfile.ZipFile(staging, "w", zipfile.ZIP_DEFLATED) as zf:
        for f in sorted(OUT.rglob("*")):
            if f.is_file() and f.suffix != ".pt":
                zf.write(f, f.relative_to(WORK))
    staging.replace(WORK / "review_results.zip")


FAILURES = []


def tail(text, n=30):
    return "\n".join([line for line in text.splitlines() if line.strip()][-n:])


def failed(where, detail):
    """Record a failed step loudly, save it, and let the run continue."""
    FAILURES.append({"where": where, "detail": detail})
    print(f"\n!!!!!! FAILED: {where}\n{detail}\n!!!!!! recorded; the run continues\n", flush=True)
    try:
        (OUT / "FAILURES.txt").write_text(
            "\n\n".join(f"== {f['where']}\n{f['detail']}" for f in FAILURES), encoding="utf-8")
        save_progress()
    except Exception as e:
        print("(could not write FAILURES.txt:", e, ")")


def publish_results(note, attempts=1):
    """Copy review_results.zip to the private dataset RESULTS_DATASET as a new version.

    Lets the laptop download the results unattended. Never stops the run: without the
    KAGGLE_KEY secret, or if Kaggle refuses, it says so and carries on. The key is
    never printed.
    """
    try:
        from kaggle_secrets import UserSecretsClient

        key = UserSecretsClient().get_secret("KAGGLE_KEY")
    except Exception:
        if not globals().get("_SAID_NO_KEY"):
            print("(optional backup copy is off - no KAGGLE_KEY secret. That is fine: "
                  "download the results from the link in the last cell.)")
            globals()["_SAID_NO_KEY"] = True
        return False
    env = {**os.environ, "KAGGLE_USERNAME": RESULTS_DATASET.split("/")[0], "KAGGLE_KEY": key}

    def kaggle(*args):
        return subprocess.run(["kaggle", *args], env=env, text=True, capture_output=True)

    try:
        save_progress()
        folder = WORK / "publish"
        shutil.rmtree(folder, ignore_errors=True)
        folder.mkdir()
        shutil.copy(WORK / "review_results.zip", folder / "review_results.zip")
        (folder / "dataset-metadata.json").write_text(json.dumps({
            "title": "codemix review results", "id": RESULTS_DATASET,
            "licenses": [{"name": "unknown"}]}))
        if subprocess.run(["kaggle", "--version"], env=env, capture_output=True).returncode != 0:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kaggle"],
                           capture_output=True)
        for attempt in range(1, attempts + 1):
            r = kaggle("datasets", "version", "-p", str(folder), "-m", note, "-q")
            said = (r.stdout + r.stderr).replace(key, "***")
            if r.returncode != 0 and ("404" in said or "not found" in said.lower()):
                r = kaggle("datasets", "create", "-p", str(folder), "-q")  # private by default
                said = (r.stdout + r.stderr).replace(key, "***")
            if r.returncode == 0:
                print(f"[results copied to {RESULTS_DATASET}: {note}]", flush=True)
                return True
            if attempt < attempts:  # usually: the previous copy is still being processed
                time.sleep(90)
        print(f"(could not copy results ({note}): {said[-300:]})", flush=True)
    except Exception as e:
        print("(could not copy results:", str(e).replace(key, "***")[:200], ")", flush=True)
    return False


# ---- the Stage-1 checkpoint every S2 run starts from, verified by its LFS hash ----
BASELINE = REPO / "checkpoints" / "baseline" / "best.pt"
# git stores the LFS pointer (with the file's sha256) whether or not the file was fetched
pointer = subprocess.run(["git", "cat-file", "-p", "HEAD:checkpoints/baseline/best.pt"],
                         cwd=REPO, text=True, capture_output=True).stdout
want = re.search(r"oid sha256:([0-9a-f]{64})", pointer)
if not want:
    raise SystemExit("checkpoints/baseline/best.pt is not tracked by LFS -- unexpected repo state")
want = want.group(1)


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for block in iter(lambda: fh.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()


source = None
for candidate in glob.glob("/kaggle/input/**/checkpoints/baseline/best.pt", recursive=True):
    if os.path.getsize(candidate) > 100_000_000 and sha256(candidate) == want:
        shutil.copy(candidate, BASELINE)
        source = candidate
        break
if source is None:
    print("Stage-1 checkpoint not found in the inputs; fetching it with git lfs ...")
    sh("git lfs install --local && git lfs pull --include checkpoints/baseline/best.pt", check=False)
    source = "git lfs"
if BASELINE.stat().st_size < 100_000_000 or sha256(BASELINE) != want:
    raise SystemExit("The Stage-1 checkpoint could not be obtained. Attach the dataset "
                     "saikrishnareddy9/codemix-s3-native-results (it contains it) and run again.")
print(f"Stage-1 checkpoint OK ({BASELINE.stat().st_size / 1e6:.0f} MB, sha256 verified) from {source}")

sh("git log --oneline -1")
sh(f"{sys.executable} -m pip install -q 'transformers>=4.57,<5' soundfile==0.12.1 librosa==0.11.0",
   check=False)

import pandas as pd
import yaml
from src.data.scoring_manifests import missing_files

channel = [p for p in glob.glob("/kaggle/input/**/channel/clips", recursive=True) if os.listdir(p)]
if not channel:
    raise SystemExit("Attach the dataset saikrishnareddy9/codemix-bundle-normalised (Add Input).")
ROOT = str(Path(channel[0]).parent)
print("data root:", ROOT)


def make_config(name, base_config, seed, options=None):
    """A copy of a published config with only the seed, out_dir and opt-in options changed."""
    cfg = yaml.safe_load((REPO / base_config).read_text())
    cfg.update({"seed": int(seed), "out_dir": f"checkpoints/review/{name}",
                "wandb_mode": "disabled"})
    cfg.update(options or {})
    path = REPO / "configs" / f"review_{name}.yaml"
    path.write_text(yaml.safe_dump(cfg, sort_keys=False))
    return str(path.relative_to(REPO)), cfg


manifests = set(EVALS.values())
for base_config, _ in BASE.values():
    cfg = yaml.safe_load((REPO / base_config).read_text())
    manifests |= {cfg["train_manifest"], cfg["dev_manifest"]}
for manifest in sorted(manifests):
    gone = missing_files(pd.read_csv(manifest), ROOT)
    print(f"  {manifest:58s} missing {len(gone)}")
    assert not gone, f"{manifest} has clips not in the dataset: {gone[:3]}"
print("every clip every run needs is present")

sh(f"{sys.executable} -m pytest tests/test_splits.py tests/test_config_guard.py "
   "tests/test_augment.py tests/test_layer_mix_and_sampler.py -q")

## 3. Run every block

In [ ]:
DEV_LINE = re.compile(r"best dev EER = ([0-9.]+)")
RUNS = []           # one row per run, saved to runs.json after each


def score(tag, ckpt, manifest, encoder="wav2vec2-base"):
    args = [sys.executable, "-m", "src.training.evaluate", "--checkpoint", ckpt, "--manifest", manifest,
            "--encoder", encoder, "--data-root", ROOT, *SCORE_ARGS.split(),
            "--scores-out", OUT / f"{tag}_scores.csv", "--out", OUT / f"{tag}.json"]
    if SMOKE:
        args += ["--limit", "200"]
    code_, text = run_logged(args)
    try:
        pooled = json.loads((OUT / f"{tag}.json").read_text())["pooled"] if code_ == 0 else None
    except Exception:
        pooled = None
    if not pooled or "eer" not in pooled:
        failed(f"scoring {tag} (exit {code_})", tail(text))
        return None
    return pooled["eer"]


def run_job(name, block, system, seed, options=None):
    """Train one config, score it on the three test sets, record and save the row."""
    t0 = time.time()
    row = {"name": name, "block": block, "system": system, "seed": seed,
           "options": options or {}, "dev_eer": None, "status": "failed"}
    try:
        base_config = BASE[system][0]
        cfg_path, cfg = make_config(name, base_config, seed, options)
        guard = sh(f"{sys.executable} -m src.training.config_guard {cfg_path}", check=False)
        if guard.returncode:
            failed(f"config_guard refused {name}", (guard.stdout + guard.stderr)[-2500:])
            return row
        code_, text = run_logged([sys.executable, "-m", "src.training.train", "--config", cfg_path,
                                  "--data-root", ROOT, "--device", "cuda"]
                                 + (["--smoke"] if SMOKE else []))
        ckpt = REPO / cfg["out_dir"] / "best.pt"
        if code_ and "model collapsed" in text:
            row["status"] = "collapsed"
            if not ckpt.is_file():
                return row
        elif code_:
            failed(f"training {name} (exit {code_})", tail(text))
            return row
        dev = DEV_LINE.findall(text)
        row["dev_eer"] = float(dev[-1]) if dev else None
        for ev, manifest in EVALS.items():
            row[ev] = score(f"{name}__{ev}", ckpt, manifest)
        if row["status"] != "collapsed":
            row["status"] = "ok" if all(row.get(ev) is not None for ev in EVALS) else "partial"
        (REPO / cfg["out_dir"] / "last.pt").unlink(missing_ok=True)  # resume file, not needed
    except Exception:
        failed(f"{name}: unexpected error", traceback.format_exc())
    finally:
        row["minutes"] = round((time.time() - t0) / 60, 1)
        RUNS.append(row)
        (OUT / "runs.json").write_text(json.dumps(RUNS, indent=2))
        save_progress()
        shown = "  ".join(f"{ev} {100 * row[ev]:.2f}" for ev in EVALS if row.get(ev) is not None)
        dev_text = "-" if row["dev_eer"] is None else f"{100 * row['dev_eer']:.2f}"
        print(f"\n[{name}] {row['status']}  dev {dev_text}  {shown}  ({row['minutes']} min)\n",
              flush=True)
    return row


def pick_variant():
    """The pre-registered rule: lowest dev EER; ties within TIE go to fewer options."""
    pool = [r for r in PRIOR + RUNS if r["status"] == "ok" and r["dev_eer"] is not None
            and (r["block"] == "variants" or (r["block"] == "deployed_seeds" and r["seed"] == 1234))]
    if not pool:
        return None
    best = min(r["dev_eer"] for r in pool)
    tied = [r for r in pool if r["dev_eer"] <= best + TIE]
    return min(tied, key=lambda r: (len(r["options"]), r["dev_eer"]))


seeds = SEEDS[:1] if SMOKE else SEEDS
CHOSEN = None
for block in BLOCKS:
    print(f"\n==================== {block} ====================\n", flush=True)
    if block == "deployed_seeds":
        for seed in [1234] + seeds:
            run_job(f"deployed_s{seed}", block, "deployed", seed)
    elif block == "variants":
        for vname, options in VARIANTS.items():
            run_job(f"deployed_{vname}_s1234", block, "deployed", 1234, options)
    elif block == "s3_seeds":
        for seed in seeds:
            run_job(f"s3xtts_s{seed}", block, "s3_xtts_channel", seed)
    elif block == "xtts_channel_seeds":
        for seed in seeds:
            run_job(f"xttsch_s{seed}", block, "xtts_channel", seed)
    elif block == "best_variant_seeds":
        CHOSEN = pick_variant()
        if CHOSEN is None:
            print("No completed published-recipe or variant run to choose from; skipped.")
        elif not CHOSEN["options"]:
            print("The published recipe has the lowest dev EER: no variant improves on it on dev,"
                  " so there is no variant to repeat. (This is itself the result.)")
        else:
            print(f"Dev-selected variant: {CHOSEN['name']} (dev EER {100 * CHOSEN['dev_eer']:.2f})")
            for seed in seeds[:2]:
                run_job(f"{CHOSEN['name'].rsplit('_s', 1)[0]}_s{seed}", block, "deployed", seed,
                        CHOSEN["options"])
    elif block == "explore_seeds":
        if EXPLORE not in VARIANTS:
            failed(f"EXPLORE={EXPLORE!r} is not one of VARIANTS", "check the settings cell")
        else:
            for seed in seeds:
                run_job(f"deployed_{EXPLORE}_s{seed}", block, "deployed", seed, VARIANTS[EXPLORE])
    else:
        failed(f"unknown block {block!r}", "check BLOCKS in the settings cell")
    if block != BLOCKS[-1]:  # the final copy is made by the summary cell
        publish_results(f"after block {block}")
print("\nALL BLOCKS FINISHED")

## 4. Summary: seeds, variants, verdict

In [ ]:
import statistics as st

lines = []
try:
    def published(prefix):
        out = {}
        for ev in EVALS:
            p = REPO / f"{prefix}__{ev}.json"
            if p.is_file():
                out[ev] = json.loads(p.read_text())["pooled"]["eer"]
        return out

    def seed_table(system, block, name_prefix):
        rows = [r for r in RUNS if r["block"] == block and r["status"] == "ok"]
        if system == "deployed":
            rows = [r for r in RUNS if r["block"] == "deployed_seeds" and r["status"] == "ok"]
        values = {ev: [r[ev] for r in rows] for ev in EVALS}
        pub = published(BASE[system][1])
        if system != "deployed":  # the published seed-1234 run joins the new seeds
            for ev in EVALS:
                if ev in pub:
                    values[ev] = [pub[ev]] + values[ev]
        n = len(next(iter(values.values()))) if values else 0
        lines.append(f"\n{name_prefix}  ({n} seeds)")
        for ev in EVALS:
            v = [100 * x for x in values[ev]]
            if len(v) >= 2:
                lines.append(f"  {ev:9s} mean {st.mean(v):6.2f}  sd {st.stdev(v):5.2f}  "
                             f"min {min(v):6.2f}  max {max(v):6.2f}   published 1234: "
                             f"{100 * pub[ev]:.2f}" if ev in pub else f"  {ev:9s} mean {st.mean(v):6.2f}")
            elif v:
                lines.append(f"  {ev:9s} {v[0]:6.2f} (one run)")

    lines.append("RUN HEALTH: " + ("every step ran without error." if not FAILURES else
                                   f"{len(FAILURES)} step(s) FAILED -- see FAILURES.txt"))
    lines.append(f"mode: {'SMOKE (numbers mean nothing)' if SMOKE else 'full'}   commit: {COMMIT}")
    lines.append("\nEER (%) per run")
    lines.append(f"{'run':34s}{'status':>10s}{'dev':>8s}" + "".join(f"{ev:>11s}" for ev in EVALS))
    for r in RUNS:
        cells = "".join(f"{'-' if r.get(ev) is None else format(100 * r[ev], '.2f'):>11s}" for ev in EVALS)
        dev = "-" if r["dev_eer"] is None else f"{100 * r['dev_eer']:.2f}"
        lines.append(f"{r['name']:34s}{r['status']:>10s}{dev:>8s}{cells}")

    if "deployed_seeds" in BLOCKS:
        seed_table("deployed", "deployed_seeds", "DEPLOYED S2 (published recipe), seeds incl. 1234 re-run")
    if "s3_seeds" in BLOCKS:
        seed_table("s3_xtts_channel", "s3_seeds", "S3 XTTS-only phone line, published 1234 + new seeds")
    if "xtts_channel_seeds" in BLOCKS:
        seed_table("xtts_channel", "xtts_channel_seeds", "S2 XTTS-only phone line, published 1234 + new seeds")

    if "variants" in BLOCKS:
        choice = pick_variant()
        lines.append("\nVARIANT SELECTION (dev EER only; test sets never used to choose)")
        for r in PRIOR + RUNS:
            if r["block"] == "variants" or (r["block"] == "deployed_seeds" and r["seed"] == 1234):
                dev = "-" if r["dev_eer"] is None else f"{100 * r['dev_eer']:.2f}"
                mark = "  <- chosen" if choice is not None and r["name"] == choice["name"] else ""
                earlier = "  (earlier run)" if r.get("earlier") else ""
                lines.append(f"  {r['name']:34s} dev {dev}{mark}{earlier}")
except Exception:
    failed("summary: unexpected error", traceback.format_exc())

text = "\n".join(lines)
print(text)
(OUT / "summary.txt").write_text(text, encoding="utf-8")
(OUT / "summary.json").write_text(json.dumps({
    "mode": "smoke" if SMOKE else "full", "commit": COMMIT, "branch": BRANCH,
    "blocks": BLOCKS, "seeds": SEEDS, "variants": VARIANTS, "tie": TIE,
    "chosen_variant": None if CHOSEN is None else CHOSEN["name"],
    "runs": RUNS, "failures": FAILURES}, indent=2, default=str))
save_progress()
publish_results("final", attempts=6)

## 5. Download

In [ ]:
import os
import zipfile
from pathlib import Path

from IPython.display import FileLink, display

WORKING = Path("/kaggle/working")
ZIP = WORKING / "review_results.zip"
os.chdir(WORKING)
if not ZIP.is_file():
    print("No results zip yet -- the cells above must run first.")
else:
    with zipfile.ZipFile(ZIP) as zf:
        count, broken = len(zf.namelist()), zf.testzip()
    print(f"review_results.zip: {count} files, {ZIP.stat().st_size / 1e6:.1f} MB,",
          "checked OK" if broken is None else f"DAMAGED at {broken} -- run this cell again")
    print("\nCLICK THIS LINK TO DOWNLOAD THE RESULTS:")
    display(FileLink("review_results.zip"))
    print("\nIf clicking does not download it: open the right-hand panel -> Output ->",
          "/kaggle/working -> review_results.zip -> the three dots -> Download.")
    print("Download it BEFORE closing this tab: the session's files are deleted when it ends.")
if globals().get("FAILURES"):
    print(f"\n!!! {len(FAILURES)} step(s) FAILED -- see FAILURES.txt in the zip and RUN HEALTH above.")